# Reddit Kolkata analytics

Runs SQL queries directly against the partitioned Parquet files under `reddit_kolkata/parquet/` via DuckDB -- no data is imported into a database file. Set the parameters in the cell below, then run all cells.

Before running this notebook for the first time (or after new data is collected), convert the latest `processed/*.csv` to Parquet:
```
python -m reddit_kolkata.collection_code.convert_to_parquet --dataset both
```

In [3]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
REDDIT_KOLKATA_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
PARQUET_DIR = REDDIT_KOLKATA_DIR / "parquet"
EXPORT_DIR = REDDIT_KOLKATA_DIR / "processed" / "notebook_exports"
POSTS_PARQUET_DIR = PARQUET_DIR / "posts"
COMMENTS_PARQUET_DIR = PARQUET_DIR / "comments"

In [4]:
# ===== PARAMETERS =====
SUBREDDIT = "kolkata"                          # a subreddit slug present in the data, or None for "all subreddits"
START_DATE = "2025-04-01"                       # inclusive, YYYY-MM-DD
END_DATE = "2025-12-31"                         # inclusive, YYYY-MM-DD
KEYWORDS = ["BJP", "TMC", "election"]           # case-insensitive keyword search list

In [5]:
import duckdb

con = duckdb.connect()  # in-memory connection only -- no .duckdb file is ever created

con.execute(f"""
    CREATE OR REPLACE VIEW posts AS
    SELECT * FROM read_parquet('{POSTS_PARQUET_DIR}/**/*.parquet', hive_partitioning=1)
""")
con.execute(f"""
    CREATE OR REPLACE VIEW comments AS
    SELECT * FROM read_parquet('{COMMENTS_PARQUET_DIR}/**/*.parquet', hive_partitioning=1)
""")

con.execute("SELECT subreddit, COUNT(*) AS rows FROM posts GROUP BY subreddit ORDER BY subreddit").df()

,subreddit,rows
0,kolkata,91328
1,kolkatacity,13499


In [6]:
def export_to_csv(df, filename: str) -> Path:
    """Export a query result DataFrame to processed/notebook_exports/<filename>."""
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    out_path = EXPORT_DIR / filename
    df.to_csv(out_path, index=False)
    print(f"Exported {len(df)} rows -> {out_path}")
    return out_path

## (a) Total number of posts in each subreddit

In [7]:
total_posts = con.execute("""
    SELECT subreddit, COUNT(*) AS total_posts
    FROM posts
    GROUP BY subreddit
    ORDER BY subreddit
""").df()
total_posts

,subreddit,total_posts
0,kolkata,91328
1,kolkatacity,13499


## (b) Monthly number of posts in each subreddit

In [8]:
monthly_posts = con.execute("""
    SELECT subreddit, strftime(date_posted, '%Y-%m') AS month, COUNT(*) AS num_posts
    FROM posts
    GROUP BY subreddit, strftime(date_posted, '%Y-%m')
    ORDER BY subreddit, month
""").df()
monthly_posts

,subreddit,month,num_posts
0,kolkata,2025-04,400
1,kolkata,2025-05,4101
2,kolkata,2025-06,4043
3,kolkata,2025-07,4381
4,kolkata,2025-08,4681
5,kolkata,2025-09,5652
6,kolkata,2025-10,4937
7,kolkata,2025-11,4496
8,kolkata,2025-12,5406
9,kolkata,2026-01,5315


## (c) Number of unique authors in each subreddit

Counts distinct authors across posts *and* comments per subreddit, matching the definition already used by `collection_code/validate.py::validate_subreddit`.

In [9]:
unique_authors = con.execute("""
    SELECT subreddit, COUNT(DISTINCT author) AS unique_authors
    FROM (
        SELECT subreddit, author FROM posts WHERE author IS NOT NULL
        UNION
        SELECT subreddit, author FROM comments WHERE author IS NOT NULL
    ) combined
    GROUP BY subreddit
    ORDER BY subreddit
""").df()
unique_authors

,subreddit,unique_authors
0,kolkata,71883
1,kolkatacity,38808


## (d) Number of authors who posted in both subreddits

The generalized query below works for any number of subreddits (no hardcoded names), so it keeps working after a third subreddit is added. A literal two-subreddit `INTERSECT` form is shown underneath as the direct equivalent for exactly two subreddits.

In [10]:
authors_in_all_subreddits = con.execute("""
    SELECT author, COUNT(DISTINCT subreddit) AS subreddits_posted_in
    FROM posts
    WHERE author IS NOT NULL
    GROUP BY author
    HAVING COUNT(DISTINCT subreddit) = (SELECT COUNT(DISTINCT subreddit) FROM posts)
    ORDER BY author
""").df()
print(f"Authors active in every currently-present subreddit: {len(authors_in_all_subreddits)}")
authors_in_all_subreddits.head(20)

Authors active in every currently-present subreddit: 2247


,author,subreddits_posted_in
0,0pointr,2
1,0xInfection,2
2,123BalBoy,2
3,1_null_,2
4,1weirdporn,2
5,2708912804tg,2
6,2Dark4U2,2
7,360tutor,2
8,4_log,2
9,4rkh4ng31,2


In [11]:
# Literal two-subreddit INTERSECT form -- assumes exactly two subreddits are present.
# If a third subreddit is added later, use the generalized GROUP BY/HAVING query above instead.
subs = con.execute("SELECT DISTINCT subreddit FROM posts ORDER BY subreddit").df()["subreddit"].tolist()

if len(subs) == 2:
    both_df = con.execute(
        """
        SELECT author FROM posts WHERE subreddit = ? AND author IS NOT NULL
        INTERSECT
        SELECT author FROM posts WHERE subreddit = ? AND author IS NOT NULL
        """,
        [subs[0], subs[1]],
    ).df()
    print(f"Authors who posted in both {subs[0]} and {subs[1]}: {len(both_df)}")
else:
    both_df = None
    print(f"{len(subs)} subreddits present ({subs}) -- INTERSECT form only applies to exactly 2; use the generalized query above.")
both_df

Authors who posted in both kolkata and kolkatacity: 2247


,author
0,thearinpaul
1,boomstar15
2,subbusss
3,Danksspy
4,Sujoy__Paul
...,...
2242,xioneeee
2243,Solid-Meet-8901
2244,AdditionalVersion315
2245,HeftyWeekend2400


## (e) Top 20 highest-scoring posts in each subreddit

In [12]:
top_scoring = con.execute("""
    SELECT subreddit, post_id, title, author, score, num_comments, date_posted, permalink
    FROM posts
    QUALIFY ROW_NUMBER() OVER (PARTITION BY subreddit ORDER BY score DESC) <= 20
    ORDER BY subreddit, score DESC
""").df()
top_scoring

,subreddit,post_id,title,author,score,num_comments,date_posted,permalink
0,kolkata,t3_1u4llmi,OP took her parents to Europe,Former_Umpire598,9240,495,2026-06-13 14:06:23+05:30,https://www.reddit.com/r/kolkata/comments/1u4l...
1,kolkata,t3_1n0gv7s,Phone repair shop in Kolkata leaked my private...,Prior_Jackfruit4428,5767,1036,2025-08-26 14:38:18+05:30,https://www.reddit.com/r/kolkata/comments/1n0g...
2,kolkata,t3_1uvz5xe,A glimpse of the Royal Bengal Tiger in the Sun...,TritiyaPandav,5654,67,2026-07-14 10:37:17+05:30,https://www.reddit.com/r/kolkata/comments/1uvz...
3,kolkata,t3_1nuehr7,Shreya Ghosal performs Indian National Anthem ...,InterestingFormal623,5308,85,2025-09-30 19:50:21+05:30,https://www.reddit.com/r/kolkata/comments/1nue...
4,kolkata,t3_1pm7mq4,"Not a fan of Rahul Gandhi ,but mad respect to ...",Fvck_trust_009,4758,196,2025-12-14 12:18:50+05:30,https://www.reddit.com/r/kolkata/comments/1pm7...
5,kolkata,t3_1plo2gr,I’ve never felt more embarrassed at Kolkata in...,Quirky_Appearance539,4512,303,2025-12-13 21:01:38+05:30,https://www.reddit.com/r/kolkata/comments/1plo...
6,kolkata,t3_1tn8m4p,I don't think most of us have any idea how uni...,Few-Reveal6853,4434,147,2026-05-25 18:35:12+05:30,https://www.reddit.com/r/kolkata/comments/1tn8...
7,kolkata,t3_1oqs7if,Same sex couple gets married in Sunderban with...,Raja_Gareebchandra,4418,373,2025-11-07 17:13:27+05:30,https://www.reddit.com/r/kolkata/comments/1oqs...
8,kolkata,t3_1kzyhjb,No One Deserves Death Threats for Words,Yeet_TheVoid,4287,588,2025-05-31 20:02:31+05:30,https://www.reddit.com/r/kolkata/comments/1kzy...
9,kolkata,t3_1t3i4q9,This is so scary. My hands are still shaking.,Strawberrycharm3,3972,720,2026-05-04 19:13:54+05:30,https://www.reddit.com/r/kolkata/comments/1t3i...


## (f) Top 20 most-commented posts in each subreddit

In [13]:
top_commented = con.execute("""
    SELECT subreddit, post_id, title, author, num_comments, score, date_posted, permalink
    FROM posts
    QUALIFY ROW_NUMBER() OVER (PARTITION BY subreddit ORDER BY num_comments DESC) <= 20
    ORDER BY subreddit, num_comments DESC
""").df()
top_commented

,subreddit,post_id,title,author,num_comments,score,date_posted,permalink
0,kolkata,t3_1r1y9m4,Let's find our school twin! 🤓Which school did ...,alt-any-lack,1950,589,2026-02-11 19:35:42+05:30,https://www.reddit.com/r/kolkata/comments/1r1y...
1,kolkata,t3_1t36883,West Bengal Election Megathread,Nghtcrwlrr,1703,182,2026-05-04 08:58:08+05:30,https://www.reddit.com/r/kolkata/comments/1t36...
2,kolkata,t3_1spjvv6,Since every city is doing this.... Why not kol...,Lonely_Spell9563,1687,293,2026-04-19 10:47:17+05:30,https://www.reddit.com/r/kolkata/comments/1spj...
3,kolkata,t3_1qtwnwc,"let's find our neighbours, our area twin",Any-Lack-1445,1229,347,2026-02-02 20:04:33+05:30,https://www.reddit.com/r/kolkata/comments/1qtw...
4,kolkata,t3_1n0gv7s,Phone repair shop in Kolkata leaked my private...,Prior_Jackfruit4428,1036,5767,2025-08-26 14:38:18+05:30,https://www.reddit.com/r/kolkata/comments/1n0g...
5,kolkata,t3_1vrpj3d,pedophile teacher at a school,MonaThe,983,3621,2026-08-18 19:11:30.099000+05:30,https://www.reddit.com/r/kolkata/comments/1vrp...
6,kolkata,t3_1rh8aj8,Let’s keep it that way.,SiriusCyrus,952,171,2026-02-28 22:51:22+05:30,https://www.reddit.com/r/kolkata/comments/1rh8...
7,kolkata,t3_1om76d4,"Found this near my compound, what animal is th...",None,755,1470,2025-11-02 08:58:15+05:30,https://www.reddit.com/r/kolkata/comments/1om7...
8,kolkata,t3_1t3i4q9,This is so scary. My hands are still shaking.,Strawberrycharm3,720,3972,2026-05-04 19:13:54+05:30,https://www.reddit.com/r/kolkata/comments/1t3i...
9,kolkata,t3_1t4npvs,Welcome to jungle raaj,jnntwrsi,677,1694,2026-05-05 23:47:41+05:30,https://www.reddit.com/r/kolkata/comments/1t4n...


## (g) Posts containing specified keywords within a date range

Uses `SUBREDDIT`, `START_DATE`, `END_DATE`, `KEYWORDS` from the parameters cell above. Every user-supplied value (keywords, dates, subreddit) is bound through DuckDB's `?` parameter list -- never string-interpolated into the SQL text -- so free-text keyword input can't break or inject into the query; only the *count* of `?` placeholders depends on `len(KEYWORDS)`.

In [14]:
def search_posts_by_keywords(subreddit, start_date, end_date, keywords):
    clauses, params = [], [start_date, end_date]
    for kw in keywords:
        clauses.append("(title ILIKE ? OR post_text ILIKE ?)")
        pattern = f"%{kw}%"
        params.extend([pattern, pattern])
    where_keywords = " OR ".join(clauses) if clauses else "FALSE"

    sql = f"""
        SELECT subreddit, post_id, title, author, date_posted, score, num_comments, permalink
        FROM posts
        WHERE date_posted >= ?
          AND date_posted < CAST(? AS DATE) + INTERVAL 1 DAY
          AND ({where_keywords})
    """
    if subreddit:
        sql += " AND subreddit = ?"
        params.append(subreddit)
    sql += " ORDER BY date_posted"

    return con.execute(sql, params).df()

keyword_results = search_posts_by_keywords(SUBREDDIT, START_DATE, END_DATE, KEYWORDS)
print(f"{len(keyword_results)} posts matched {KEYWORDS} between {START_DATE} and {END_DATE} (subreddit={SUBREDDIT!r})")
keyword_results.head(20)

615 posts matched ['BJP', 'TMC', 'election'] between 2025-04-01 and 2025-12-31 (subreddit='kolkata')


,subreddit,post_id,title,author,date_posted,score,num_comments,permalink
0,kolkata,t3_1jom25t,Bengal BJP accuses Mamata of insulting Hindu r...,adorepoems06,2025-04-01 08:50:24+05:30,144,167,https://www.reddit.com/r/kolkata/comments/1jom...
1,kolkata,t3_1jpelhq,Meet 'Sadiya Akhtar' from Kolkata She wants al...,Specific_Engine_8356,2025-04-02 08:24:47+05:30,12,5,https://www.reddit.com/r/kolkata/comments/1jpe...
2,kolkata,t3_1jpgb5m,Industrialisation in west bengal,Dry_Chicken_7387,2025-04-02 09:58:31+05:30,1,0,https://www.reddit.com/r/kolkata/comments/1jpg...
3,kolkata,t3_1jphf0p,Thinking fallacies,LingoNerd64,2025-04-02 11:10:18+05:30,8,10,https://www.reddit.com/r/kolkata/comments/1jph...
4,kolkata,t3_1jq4mwk,How a Pro-BJP Village in Junglemahal Mounted a...,sleepless-deadman,2025-04-03 06:06:07+05:30,0,0,https://www.reddit.com/r/kolkata/comments/1jq4...
5,kolkata,t3_1jqchg7,Union Govt wants non-academics to be allowed t...,sleepless-deadman,2025-04-03 13:20:05+05:30,7,2,https://www.reddit.com/r/kolkata/comments/1jqc...
6,kolkata,t3_1jqecc0,"25,000 People Lost Their Jobs Overnight—IDK BU...",Significant_Use_4246,2025-04-03 15:25:12+05:30,118,176,https://www.reddit.com/r/kolkata/comments/1jqe...
7,kolkata,t3_1jqjnb5,How many of you witnessed local leaders taking...,Annonymous_7,2025-04-03 19:48:04+05:30,34,28,https://www.reddit.com/r/kolkata/comments/1jqj...
8,kolkata,t3_1jr3e92,Animation studio and Art culture in Kolkata,Character-Ad-1829,2025-04-04 09:55:49+05:30,3,3,https://www.reddit.com/r/kolkata/comments/1jr3...
9,kolkata,t3_1kdbqfp,Selection into Scottish Church College,None,2025-05-03 02:30:44+05:30,1,0,https://www.reddit.com/r/kolkata/comments/1kdb...


## (h) Top 20 most frequently linked external domains in each subreddit

In [15]:
top_domains = con.execute("""
    SELECT subreddit, external_domain, COUNT(*) AS link_count
    FROM posts
    WHERE external_domain IS NOT NULL AND external_domain <> ''
    GROUP BY subreddit, external_domain
    QUALIFY ROW_NUMBER() OVER (PARTITION BY subreddit ORDER BY COUNT(*) DESC) <= 20
    ORDER BY subreddit, link_count DESC
""").df()
top_domains

,subreddit,external_domain,link_count
0,kolkata,youtu.be,8
1,kolkata,x.com,2
2,kolkata,www.telegraphindia.com,2
3,kolkata,www.festivalofindependence.com,1
4,kolkata,link.deezer.com,1
5,kolkata,kitnagaz.in,1
6,kolkata,v.redd.it,1
7,kolkata,tv9bangla.com,1
8,kolkata,sommukhopadhyay.blogspot.com,1
9,kolkata,www.thehindu.com,1


## Export example

Any of the DataFrames above can be exported the same way; the keyword-search result is used here as the demonstration.

In [16]:
export_to_csv(keyword_results, "keyword_search_results.csv")

Exported 615 rows -> /Users/harshalshah05/Documents/4-1 SEM/Project/reddit_kolkata/processed/notebook_exports/keyword_search_results.csv


PosixPath('/Users/harshalshah05/Documents/4-1 SEM/Project/reddit_kolkata/processed/notebook_exports/keyword_search_results.csv')